In [1]:
import mlflow
import os
from mlflow.tracking import MlflowClient

print("="*50)
print("MLFLOW SETUP VERIFICATION")
print("="*50)

# 1. Check tracking URI
print(f"\n1. Current tracking URI: {mlflow.get_tracking_uri()}")

# 2. Set tracking URI explicitly
mlflow.set_tracking_uri("http://localhost:5000")
print(f"2. Set tracking URI to: {mlflow.get_tracking_uri()}")

# 3. Check if server is reachable
try:
    import requests
    response = requests.get("http://localhost:5000/health")
    print("3. ✅ MLflow server is reachable")
except:
    print("3. ❌ Cannot reach MLflow server - make sure it's running")
    exit()

# 4. List existing experiments
client = MlflowClient()
experiments = client.search_experiments()
print(f"\n4. Found {len(experiments)} experiment(s):")
for exp in experiments:
    print(f"   - {exp.name} (ID: {exp.experiment_id})")

# 5. Create a new test experiment
test_exp_name = "Test_Experiment_Clean"
print(f"\n5. Creating test experiment: {test_exp_name}")

try:
    exp_id = client.create_experiment(test_exp_name)
    print(f"   ✅ Created with ID: {exp_id}")
except Exception as e:
    print(f"   ⚠️  Already exists or error: {e}")
    exp = client.get_experiment_by_name(test_exp_name)
    exp_id = exp.experiment_id

# 6. Run a test experiment
print(f"\n6. Running test experiment...")
mlflow.set_experiment(test_exp_name)

with mlflow.start_run(run_name="Test_Run_1"):
    mlflow.log_param("test_param", "hello_world")
    mlflow.log_metric("test_metric", 123.45)
    
    # Create a simple artifact
    with open("test_artifact.txt", "w") as f:
        f.write("This is a test artifact")
    mlflow.log_artifact("test_artifact.txt")
    
    print(f"   ✅ Run completed!")
    print(f"   Run ID: {mlflow.active_run().info.run_id}")

# 7. Verify the run was saved
print(f"\n7. Verifying run saved...")
runs = client.search_runs(exp_id)
print(f"   Found {len(runs)} run(s) in experiment")

for run in runs:
    print(f"   Run: {run.info.run_name}")
    print(f"     - Params: {run.data.params}")
    print(f"     - Metrics: {run.data.metrics}")
    print(f"     - Artifact URI: {run.info.artifact_uri}")

# 8. Check where files are stored
print(f"\n8. Checking file system storage...")
current_dir = os.getcwd()
print(f"   Current directory: {current_dir}")

# Check for mlruns folder
mlruns_path = os.path.join(current_dir, "mlruns")
if os.path.exists(mlruns_path):
    print(f"   ✅ mlruns folder exists at: {mlruns_path}")
    # List contents
    for item in os.listdir(mlruns_path):
        if item != ".trash":
            print(f"      - {item}")
else:
    print(f"   ❌ mlruns folder NOT found")

# Check for artifacts
artifacts_path = os.path.join(current_dir, "mlflow-artifacts")
if os.path.exists(artifacts_path):
    print(f"   ✅ mlflow-artifacts folder exists at: {artifacts_path}")
else:
    print(f"   ⚠️  mlflow-artifacts folder not yet created (normal if no artifacts)")

# 9. Check the UI
print(f"\n9. View results in browser:")
print(f"   👉 Open: http://localhost:5000")
print(f"   Look for experiment: {test_exp_name}")

print("\n" + "="*50)
print("VERIFICATION COMPLETE!")
print("="*50)

d:\Interview\MLOPS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLFLOW SETUP VERIFICATION

1. Current tracking URI: sqlite:///D:/Interview/MLOPS/1.%20MLFOW/mlflow.db
2. Set tracking URI to: http://localhost:5000
3. ✅ MLflow server is reachable

4. Found 1 experiment(s):
   - Default (ID: 0)

5. Creating test experiment: Test_Experiment_Clean
   ✅ Created with ID: 395762862080893211

6. Running test experiment...
   ✅ Run completed!
   Run ID: 37046f7fc2324d62958ced26822b8a60
🏃 View run Test_Run_1 at: http://localhost:5000/#/experiments/395762862080893211/runs/37046f7fc2324d62958ced26822b8a60
🧪 View experiment at: http://localhost:5000/#/experiments/395762862080893211

7. Verifying run saved...
   Found 1 run(s) in experiment
   Run: Test_Run_1
     - Params: {'test_param': 'hello_world'}
     - Metrics: {'test_metric': 123.45}
     - Artifact URI: mlflow-artifacts:/395762862080893211/37046f7fc2324d62958ced26822b8a60/artifacts

8. Checking file system storage...
   Current directory: d:\Interview\MLOPS\1. MLFOW
   ❌ mlruns folder NOT found
   ⚠️  ml

In [2]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:5000")
client = MlflowClient()

print("Checking for original experiments...")

# Search for your original experiments
experiment_names = ["My Experiment", "Check localhost connection"]

for exp_name in experiment_names:
    exp = client.get_experiment_by_name(exp_name)
    if exp:
        print(f"\n✅ Found: {exp_name} (ID: {exp.experiment_id})")
        runs = client.search_runs(exp.experiment_id)
        print(f"   Runs found: {len(runs)}")
        for run in runs:
            print(f"   - {run.info.run_name}: {run.info.status}")
            print(f"     Metrics: {run.data.metrics}")
            print(f"     Params: {run.data.params}")
    else:
        print(f"\n❌ Not found: {exp_name}")

# List ALL experiments
print("\n" + "="*50)
print("ALL EXPERIMENTS ON SERVER:")
experiments = client.search_experiments()
for exp in experiments:
    runs_count = len(client.search_runs(exp.experiment_id))
    print(f"  • {exp.name} (ID: {exp.experiment_id}) - {runs_count} runs")

Checking for original experiments...

❌ Not found: My Experiment

❌ Not found: Check localhost connection

ALL EXPERIMENTS ON SERVER:
  • Test_Experiment_Clean (ID: 395762862080893211) - 1 runs
  • Default (ID: 0) - 2 runs


In [3]:
import os
import sqlite3
from pathlib import Path

print("Searching for MLflow database files...")
for root, dirs, files in os.walk("D:\\Interview\\MLOPS"):
    for file in files:
        if file.endswith(".db") and "mlflow" in file.lower():
            full_path = os.path.join(root, file)
            size = os.path.getsize(full_path)
            print(f"\nFound: {full_path}")
            print(f"Size: {size} bytes")
            
            # Try to read experiments
            try:
                conn = sqlite3.connect(full_path)
                cursor = conn.cursor()
                cursor.execute("SELECT name FROM experiments")
                exps = cursor.fetchall()
                print(f"Experiments in this DB: {len(exps)}")
                for exp in exps:
                    print(f"  - {exp[0]}")
                conn.close()
            except:
                pass

Searching for MLflow database files...

Found: D:\Interview\MLOPS\1. MLFOW\mlflow.db
Size: 712704 bytes
Experiments in this DB: 3
  - Check localhost connection
  - Default
  - My Experiment

Found: D:\Interview\MLOPS\1.%20MLFOW\mlflow.db
Size: 712704 bytes
Experiments in this DB: 1
  - Default


In [4]:
import mlflow
from mlflow.tracking import MlflowClient

# Set tracking URI to your SQLite database
mlflow.set_tracking_uri("sqlite:///D:/Interview/MLOPS/1.%20MLFOW/mlflow.db")
client = MlflowClient()

print("="*50)
print("YOUR ORIGINAL EXPERIMENTS (from SQLite)")
print("="*50)

# Check your original experiments
exp_names = ["My Experiment", "Check localhost connection"]

for exp_name in exp_names:
    exp = client.get_experiment_by_name(exp_name)
    if exp:
        print(f"\n📊 Experiment: {exp_name}")
        print(f"   ID: {exp.experiment_id}")
        
        runs = client.search_runs(exp.experiment_id)
        print(f"   Total runs: {len(runs)}")
        
        for i, run in enumerate(runs, 1):
            print(f"\n   Run {i}: {run.info.run_name or run.info.run_id[:8]}")
            print(f"      Status: {run.info.status}")
            if run.data.params:
                print(f"      Params: {run.data.params}")
            if run.data.metrics:
                print(f"      Metrics: {run.data.metrics}")
    else:
        print(f"\n❌ Experiment '{exp_name}' not found")

# List all experiments
print("\n" + "="*50)
print("ALL EXPERIMENTS IN SQLITE:")
print("="*50)
experiments = client.search_experiments()
for exp in experiments:
    runs_count = len(client.search_runs(exp.experiment_id))
    print(f"  • {exp.name} (ID: {exp.experiment_id}) - {runs_count} runs")

YOUR ORIGINAL EXPERIMENTS (from SQLite)

❌ Experiment 'My Experiment' not found

❌ Experiment 'Check localhost connection' not found

ALL EXPERIMENTS IN SQLITE:
  • Default (ID: 0) - 0 runs


In [5]:
import sqlite3
import json

conn = sqlite3.connect("D:/Interview/MLOPS/1. MLFOW/mlflow.db")
cursor = conn.cursor()

# Get all runs with their params and metrics
cursor.execute("""
    SELECT 
        e.name as experiment,
        r.run_uuid,
        r.status,
        datetime(r.start_time/1000, 'unixepoch') as start_time
    FROM runs r
    JOIN experiments e ON r.experiment_id = e.experiment_id
    ORDER BY e.name, r.start_time
""")

print("ALL RUNS IN DATABASE:")
print("="*60)
for exp, run_id, status, start_time in cursor.fetchall():
    print(f"\n📊 Experiment: {exp}")
    print(f"   Run ID: {run_id[:8]}...")
    print(f"   Status: {status}")
    print(f"   Started: {start_time}")
    
    # Get params for this run
    cursor.execute("SELECT key, value FROM params WHERE run_uuid=?", (run_id,))
    params = cursor.fetchall()
    if params:
        print(f"   Parameters:")
        for key, value in params:
            print(f"     - {key}: {value}")
    
    # Get metrics for this run
    cursor.execute("SELECT key, value FROM metrics WHERE run_uuid=?", (run_id,))
    metrics = cursor.fetchall()
    if metrics:
        print(f"   Metrics:")
        for key, value in metrics:
            print(f"     - {key}: {value}")

conn.close()

ALL RUNS IN DATABASE:

📊 Experiment: Check localhost connection
   Run ID: 1216a30c...
   Status: FINISHED
   Started: 2026-05-20 16:07:42
   Parameters:
     - param1: 5
   Metrics:
     - metric1: 0.89

📊 Experiment: Check localhost connection
   Run ID: e7de0534...
   Status: FINISHED
   Started: 2026-05-20 16:12:05
   Parameters:
     - param1: 5
     - param2: test
   Metrics:
     - metric1: 0.89
     - metric2: 0.95

📊 Experiment: Check localhost connection
   Run ID: ada5e192...
   Status: FINISHED
   Started: 2026-05-20 16:13:07
   Metrics:
     - test1: 1.0
     - Krish1: 2.0

📊 Experiment: Check localhost connection
   Run ID: 7a1dad5e...
   Status: FINISHED
   Started: 2026-05-20 16:13:12
   Metrics:
     - test2: 1.0
     - Krish2: 2.0

📊 Experiment: My Experiment
   Run ID: c4b36f34...
   Status: FINISHED
   Started: 2026-05-20 16:22:43
   Parameters:
     - param1: 5
   Metrics:
     - metric1: 0.89

📊 Experiment: My Experiment
   Run ID: de0e1b4a...
   Status: FINISHED
